# Laboratorio 12a — Pronóstico con redes recurrentes
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 25 de noviembre de 2026 · Clase 12 · Lab. Central 114
**Bloque:** Lab A · ~10:40–11:30 (50 min), después de la teoría RNN
**Viene de:** teoría RNN (deck «Clase 12 - RNN y GAN», diapos 1–12: memoria que pasa de palabra en palabra, teléfono descompuesto, LSTM, GRU y ventanas deslizantes).
**Sigue:** descanso → teoría GAN → Lab 12b (otro notebook, ~12:20–13:00).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **A1** | Serie inventada, corte en orden y «mañana = hoy» | 10 min |
| **A2** | Ventanas deslizantes | 10 min |
| **A3** | RNN, LSTM y GRU | 20 min |
| **A4** | Pronóstico de 100 pasos y cuánta historia mirar | 10 min |

## Objetivos de aprendizaje

1. Cortar una serie de tiempo **en orden** y medir una regla simple: «mañana = hoy».
2. Convertir la serie en **ventanas** (30 valores → el siguiente) y leer la forma `(1020, 30, 1)`.
3. Entrenar una RNN simple, una LSTM y una GRU, y comprobar que las tres le ganan a «mañana = hoy».
4. Ver cómo se acumula el error al pronosticar muchos pasos y cuánta historia le conviene mirar a la red.

> Todo el notebook es **offline**: la serie se inventa aquí mismo. Funciona en Colab o en tu computador (basta la CPU).

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time, copy, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy pandas matplotlib scikit-learn torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import HTML, display
try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("No se encontró PyTorch. En tu computador: activa el entorno virtual e instala con "
                      "`pip install -r requirements.txt`. Si no, abre este notebook en Google Colab.") from e

SEED = 42
RAPIDO = False   # True = menos épocas, solo para ensayar el código (los valores del libreto usan False)
torch.set_num_threads(1)
T0 = time.time()


def fijar_semillas(semilla=SEED):
    """Deja reproducibles random, NumPy y PyTorch."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)


fijar_semillas()

# Paleta del deck «Clase 12 - RNN y GAN»
AZUL, NARANJA, MARINO, GRIS, GRIS_CLARO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#d9dbdf"
COLOR_RED = {"rnn": GRIS, "lstm": NARANJA, "gru": AZUL}          # como las válvulas de la diapo 9
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__, "| RAPIDO =", RAPIDO, "| Colab =", IN_COLAB)

## Mapa del lab

Hoy haremos exactamente lo de la clase, pero con código: una **serie** se corta en **ventanas**, una **red con memoria** lee cada ventana paso a paso y entrega un **pronóstico** del valor que sigue.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.1))
ax.set(xlim=(0, 10), ylim=(0.3, 1.7))
ax.axis("off")
cajas = [("serie", "1500 valores en orden", MARINO), ("ventanas", "30 pasos → el siguiente", NARANJA),
         ("red con memoria", "RNN · LSTM · GRU", AZUL), ("pronóstico", "el valor que sigue", GRIS)]
for k, (titulo, detalle, color) in enumerate(cajas):
    x = 0.1 + k * 2.5
    ax.add_patch(FancyBboxPatch((x, 0.5), 2.0, 1.0, boxstyle="round,pad=0.04,rounding_size=0.15", fc=color, ec="none"))
    ax.text(x + 1.0, 1.13, titulo, ha="center", va="center", color="white", fontsize=11.5, fontweight="bold")
    ax.text(x + 1.0, 0.8, detalle, ha="center", va="center", color="white", fontsize=9.5)
    if k < len(cajas) - 1:
        ax.annotate("", xy=(x + 2.47, 1.0), xytext=(x + 2.07, 1.0),
                    arrowprops=dict(arrowstyle="-|>", color=MARINO, lw=1.8))
plt.show()

---
## Parte A1 — Serie inventada, corte en orden y «mañana = hoy» (10 min)

> **En palabras simples (diapo 11).** Inventamos una serie que sube de a poco, se repite en ciclos y tiene ruido, como la demanda eléctrica de una ciudad. La cortamos en orden: lo primero para aprender, lo del medio para ajustar y lo último para la prueba final. Nunca se baraja: sería estudiar mirando las respuestas del futuro.

**Consigna**

1. Arma una serie de **1500 pasos** que sume cuatro cosas:
   - una subida suave (tendencia): `0.002 * t`;
   - un ciclo largo que se repite cada **50 pasos**, de altura 1;
   - un ciclo corto que se repite cada **7 pasos**, de altura 0,5;
   - ruido al azar de tamaño 0,2 (`np.random.normal(0, 0.2, ...)`).
2. Córtala **en orden**: 70 % para entrenar, 15 % para validar y 15 % para la prueba final.
3. Escala con el promedio y la desviación **del entrenamiento** (también para validación y prueba).
4. Mide la regla más simple posible, **«mañana = hoy»** (su nombre técnico es *línea base ingenua*): predecir que el valor siguiente es igual al de hoy.

In [ ]:
# 1) Serie inventada: tendencia + ciclo de 50 pasos + ciclo de 7 pasos + ruido
np.random.seed(0)
t = np.arange(1500)
# TODO: construye `serie` sumando las cuatro piezas de la consigna.
#       Pista para un ciclo que se repite cada P pasos: np.sin(2 * np.pi * t / P)
#       Pista para el ruido: np.random.normal(0, 0.2, size=t.size)
serie = ...

In [ ]:
# --- Autochequeo: la serie ---
assert isinstance(serie, np.ndarray), "Todavía falta completar `serie` (reemplaza los ...)."
assert serie.shape == (1500,), f"La serie debería tener 1500 valores; tiene forma {serie.shape}."
assert 0.8 < serie.std() < 1.6, "La serie se ve rara: revisa las alturas de los ciclos (1 y 0,5) y el ruido (0,2)."
assert 1.0 < serie[-100:].mean() - serie[:100].mean() < 4.0, "La serie debería subir de a poco: revisa la tendencia 0.002 * t."
print("✔ La serie está bien")

In [ ]:
# 2) Corte en orden (sin barajar)
def dividir_cronologico(serie, p_train=0.70, p_val=0.15):
    """Devuelve (train, val, test) respetando el orden del tiempo."""
    # TODO: calcula dónde cortar (al 70 % y al 85 % del largo) y devuelve los tres tramos SIN barajar.
    ...


assert dividir_cronologico(serie) is not None, "`dividir_cronologico` todavía no devuelve nada: completa el TODO (return ...)."
train, val, test = dividir_cronologico(serie)

# 3) Escalado con el promedio y la desviación SOLO del entrenamiento
# TODO: ¿de qué tramo salen el promedio y la desviación? (pista: no mires el futuro)
mu, sd = ..., ...

In [ ]:
# --- Autochequeo: el corte y el escalado ---
assert (len(train), len(val), len(test)) == (1050, 225, 225), \
    f"Los tramos deberían medir 1050, 225 y 225; miden {len(train)}, {len(val)} y {len(test)}."
assert np.allclose(np.concatenate([train, val, test]), serie), "Los tramos, pegados, deben ser la serie original y en el mismo orden."
assert mu is not ... and np.isclose(mu, train.mean()) and np.isclose(sd, train.std()), \
    "El promedio y la desviación deben salir SOLO del tramo de entrenamiento."
print("✔ Corte y escalado bien")

In [ ]:
def escalar(x):
    return (x - mu) / sd


def desescalar(x):
    return x * sd + mu


train_e, val_e, test_e = escalar(train), escalar(val), escalar(test)


def mae(y, yhat):
    """Error absoluto medio: en promedio, ¿por cuánto me equivoco?"""
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))


def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))


# La serie por tramos (entrenamiento azul marino, validación gris, prueba naranja)
fig, ax = plt.subplots(figsize=(10, 3.2))
i1, i2 = len(train), len(train) + len(val)
ax.plot(t[:i1], train, color=MARINO, lw=1, label="entrenamiento (70 %)")
ax.plot(t[i1:i2], val, color=GRIS, lw=1, label="validación (15 %)")
ax.plot(t[i2:], test, color=NARANJA, lw=1, label="prueba (15 %)")
ax.set(title="La serie inventada, cortada en orden", xlabel="paso de tiempo", ylabel="valor")
ax.legend(loc="upper left", frameon=False)
plt.show()

In [ ]:
# 4) «Mañana = hoy» en el tramo de prueba (unidades originales)
# TODO: y_real = los valores de prueba desde el segundo en adelante (lo que pasó «mañana»)
#       y_manana_hoy = los valores de prueba hasta el penúltimo (el valor de «hoy»)
y_real = ...
y_manana_hoy = ...
assert y_real is not ... and y_manana_hoy is not ..., "Completa `y_real` e `y_manana_hoy` (todavía hay ...)."
base_mae, base_rmse = mae(y_real, y_manana_hoy), rmse(y_real, y_manana_hoy)
print(f"«Mañana = hoy» en prueba:  MAE = {base_mae:.3f} | RMSE = {base_rmse:.3f}")

In [ ]:
# --- Autochequeo: «mañana = hoy» ---
assert len(y_real) == len(y_manana_hoy) == 224, "Deberían quedar 224 parejas (hoy, mañana) en el tramo de prueba de 225."
assert np.allclose(y_real[:-1], y_manana_hoy[1:]), "y_manana_hoy debe ir un paso atrasado respecto de y_real."
assert 0.25 < base_mae < 0.45, f"El MAE de «mañana = hoy» ({base_mae:.3f}) está fuera de lo esperado (≈ 0,35)."
print("✔ «Mañana = hoy» bien")

**Pregunta A1.** ¿Por qué no se baraja la serie antes de cortarla? ¿Qué le pasaría a la nota de la prueba si lo hiciéramos?

**Respuesta:** (escribe aquí)

---
## Parte A2 — Ventanas deslizantes (10 min)

> **En palabras simples (diapo 11).** La red no ve la serie entera: ve ventanas de 30 valores y tiene que adivinar el que sigue. Cada ventana es un ejemplo de entrenamiento. La forma (1020, 30, 1) se lee: 1020 ventanas, de 30 pasos, con 1 número por paso.

**Consigna.** Completa `crear_ventanas`: la ventana número `i` son los valores `i, i+1, …, i+29` y su objetivo es el valor `i+30`. Las ventanas se arman **dentro de cada tramo**, para no mezclar entrenamiento con validación o prueba.

In [ ]:
def crear_ventanas(serie_esc, ventana=30):
    """X[i] = serie[i : i+ventana]  ->  y[i] = serie[i+ventana]."""
    # TODO: arma X con forma (n_ventanas, ventana, 1) e y con el valor que sigue a cada ventana.
    #       Pista: ¿cuántas ventanas completas caben? len(serie_esc) - ventana
    #       Pista: np.stack([...]) apila una lista de ventanas; X[..., None] agrega el «1 número por paso».
    #       Devuelve X e y como float32.
    ...


VENTANA = 30
assert crear_ventanas(train_e, VENTANA) is not None, "`crear_ventanas` todavía no devuelve nada: completa el TODO (return X, y)."
Xtr, ytr = crear_ventanas(train_e, VENTANA)
Xva, yva = crear_ventanas(val_e, VENTANA)
Xte, yte = crear_ventanas(test_e, VENTANA)
print("Xtr", Xtr.shape, "| Xva", Xva.shape, "| Xte", Xte.shape)

In [ ]:
# --- Autochequeo: las ventanas ---
assert Xtr.shape == (1020, 30, 1), f"Revisa la forma: debería ser (1020, 30, 1) y es {Xtr.shape}."
assert Xva.shape == (195, 30, 1) and Xte.shape == (195, 30, 1), "Validación y prueba deberían tener forma (195, 30, 1)."
assert Xtr.dtype == np.float32 and ytr.dtype == np.float32, "Convierte X e y a float32 (.astype(np.float32))."
assert np.allclose(Xtr[1, :-1, 0], Xtr[0, 1:, 0]), "Cada ventana debería correrse un paso respecto de la anterior."
assert np.isclose(ytr[0], train_e[30]), "El objetivo de la ventana 0 debería ser el valor 30 del tramo (el que sigue)."
print("✔ Ventanas bien")

### La ventana en movimiento

Es la diapo 11, pero con los datos de este lab: la **ventana naranja** (30 pasos) recorre el inicio del tramo de entrenamiento y el **punto azul** es lo que la red tiene que adivinar. Cada cuadro es un ejemplo de entrenamiento.

In [ ]:
tramo = train_e[:150]
inicios = list(range(0, 120, 4))          # 30 cuadros: la ventana avanza de a 4 pasos

fig, ax = plt.subplots(figsize=(9, 3.1), dpi=80)
ax.plot(tramo, color=GRIS, lw=1, alpha=0.7)
sombra = ax.add_patch(Rectangle((0, -10), VENTANA - 1, 20, color=NARANJA, alpha=0.12, lw=0))
linea, = ax.plot([], [], "o-", color=NARANJA, ms=3, lw=1.8, label="ventana: los 30 pasos que ve la red")
punto, = ax.plot([], [], "o", color=AZUL, ms=10, label="el punto a adivinar")
ax.set(xlim=(-2, 152), ylim=(tramo.min() - 0.3, tramo.max() + 1.0), xlabel="paso", ylabel="valor escalado")
ax.legend(loc="upper left", frameon=False, ncol=2, fontsize=9)
fig.subplots_adjust(bottom=0.18)


def cuadro(k):
    i = inicios[k]
    linea.set_data(np.arange(i, i + VENTANA), tramo[i:i + VENTANA])
    punto.set_data([i + VENTANA], [tramo[i + VENTANA]])
    sombra.set_x(i)
    ax.set_title(f"Ventana #{i} (= Xtr[{i}]): pasos {i}–{i + VENTANA - 1} → adivinar el paso {i + VENTANA}")
    return linea, punto, sombra


anim = animation.FuncAnimation(fig, cuadro, frames=len(inicios), interval=350)
plt.close(fig)                            # evita que la figura salga repetida debajo de la animación
HTML(anim.to_jshtml())

In [ ]:
# Comprobación a mano: una ventana y su objetivo
i = 100
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(range(VENTANA), Xtr[i, :, 0], "o-", color=NARANJA, ms=3, label="entrada: 30 pasos")
ax.plot(VENTANA, ytr[i], "o", color=AZUL, ms=9, label="objetivo: el paso 31")
ax.set(title=f"Ventana #{i}", xlabel="paso dentro de la ventana", ylabel="valor escalado")
ax.legend(frameon=False)
plt.show()
print(f"Último valor de la ventana: {Xtr[i, -1, 0]:.3f} | objetivo: {ytr[i]:.3f} | debería ser train_e[{i + VENTANA}] = {train_e[i + VENTANA]:.3f}")

**Pregunta A2.** El tramo de entrenamiento tiene 1050 valores, pero salen 1020 ventanas. ¿Dónde quedaron las otras 30?

**Respuesta:** (escribe aquí)

---
## Parte A3 — RNN, LSTM y GRU (20 min)

> **En palabras simples (diapos 4, 6 y 7).** Las tres redes leen la ventana paso a paso y llevan una memoria de 32 números. La RNN simple la reescribe en cada paso; la LSTM y la GRU tienen válvulas que deciden qué borrar y qué anotar. El *clipping* es el techo de la diapo 6: si la corrección se dispara, la cortamos.

**La receta (igual para las tres redes)**

- Una capa recurrente (`nn.RNN`, `nn.LSTM` o `nn.GRU`) con memoria (estado oculto) de **32 números**, y una capa lineal que mira solo el **último** paso y entrega el pronóstico.
- Adam con `lr=3e-3`, error cuadrático (MSE), **30 épocas**, lotes de 32 ventanas y techo para la corrección (*gradient clipping*) en 1,0.
- **Nos quedamos con la mejor época.** Es la misma idea de la parada temprana de la Clase 10: al final, la red vuelve a los pesos de la época en que mejor le fue en validación.

In [ ]:
class Recurrente(nn.Module):
    """Capa recurrente (rnn | lstm | gru) + capa lineal sobre el último paso."""

    def __init__(self, tipo="lstm", memoria=32):
        super().__init__()
        # TODO: elige la clase según `tipo` (nn.RNN, nn.LSTM o nn.GRU) y crea
        #       self.rnn con input_size=1, hidden_size=memoria, num_layers=1, batch_first=True
        #       self.fc  = nn.Linear(memoria, 1)
        ...

    def forward(self, x):                              # x: (lote, pasos, 1)
        # TODO: pasa x por self.rnn, quédate con la salida del ÚLTIMO paso y aplícale self.fc.
        #       Devuelve un tensor de forma (lote,)  (pista: .squeeze(-1))
        ...


def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

In [ ]:
# --- Autochequeo: la red ---
esperados = {"rnn": 1153, "lstm": 4513, "gru": 3393}
for tipo, n_esp in esperados.items():
    n = n_parametros(Recurrente(tipo))
    assert n == n_esp, f"{tipo.upper()}: se esperaban {n_esp} parámetros y hay {n}. Revisa self.rnn y self.fc en __init__."
salida = Recurrente("lstm")(torch.zeros(4, 30, 1))
assert salida is not None, "forward todavía no devuelve nada (reemplaza los ...)."
assert salida.shape == (4,), f"Para 4 ventanas la salida debería tener forma (4,); tiene {tuple(salida.shape)}."
print("✔ La red está bien ->", esperados)

In [ ]:
EPOCAS = 8 if RAPIDO else 30


def entrenar(modelo, Xtr, ytr, Xva, yva, epocas=EPOCAS, lr=3e-3, lote=32, clip=1.0):
    """Adam + MSE con techo para la corrección. Al final restaura la mejor época (según validación)."""
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    perdida = nn.MSELoss()
    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr)
    Xva_t, yva_t = torch.tensor(Xva), torch.tensor(yva)
    hist = {"train": [], "val": []}
    mejor_val, mejor_pesos, mejor_epoca = float("inf"), None, None
    for ep in range(epocas):
        modelo.train()
        perm = torch.randperm(len(Xtr_t))
        acum = 0.0
        for i in range(0, len(perm), lote):
            idx = perm[i:i + lote]
            # TODO: un paso de entrenamiento con el lote `idx`:
            #   1) borrar las correcciones anteriores (opt.zero_grad())
            #   2) calcular el error MSE del lote y guardarlo en `l`
            #   3) calcular las correcciones (l.backward())
            #   4) poner el techo: nn.utils.clip_grad_norm_(modelo.parameters(), clip)
            #   5) aplicar la corrección (opt.step())
            l = ...
            acum += l.item() * len(idx)
        hist["train"].append(acum / len(perm))
        modelo.eval()
        with torch.no_grad():
            hist["val"].append(perdida(modelo(Xva_t), yva_t).item())
        if hist["val"][-1] < mejor_val:                # ¿es la mejor época hasta ahora?
            mejor_val, mejor_epoca = hist["val"][-1], ep
            # TODO (una línea): guarda una COPIA de los pesos actuales en `mejor_pesos`.
            #       Pista: copy.deepcopy(modelo.state_dict())  — sin la copia, se seguirían modificando.
            mejor_pesos = ...
    assert mejor_pesos is not ..., "Falta el TODO de `entrenar`: guarda una copia de los pesos en `mejor_pesos`."
    modelo.load_state_dict(mejor_pesos)                # volvemos a la mejor versión
    hist["mejor_epoca"] = mejor_epoca
    return hist

In [ ]:
def predecir(modelo, X):
    modelo.eval()
    with torch.no_grad():
        return modelo(torch.tensor(X)).numpy()


def evaluar(modelo, X, y):
    """MAE y RMSE en las unidades originales (basta multiplicar por sd: el escalado es lineal)."""
    p = predecir(modelo, X)
    return mae(y * sd, p * sd), rmse(y * sd, p * sd)


def perdida_val(modelo, X, y):
    """MSE en validación (datos escalados), igual al que registra `entrenar`."""
    return float(np.mean((predecir(modelo, X) - y) ** 2))

In [ ]:
# --- Autochequeo: `entrenar` (un ensayo corto con 4 épocas y 256 ventanas) ---
fijar_semillas(SEED)
_m = Recurrente("rnn")
try:
    _h = entrenar(_m, Xtr[:256], ytr[:256], Xva, yva, epocas=4)
except AttributeError as e:
    raise AssertionError("Falta completar el paso de entrenamiento en `entrenar` (todavía hay `l = ...`).") from e
assert len(_h["val"]) == 4, "Debería quedar una pérdida de validación por época."
assert _h["val"][-1] < _h["val"][0] or _h["mejor_epoca"] > 0, "La pérdida de validación no bajó nunca: revisa el paso de entrenamiento."
assert abs(perdida_val(_m, Xva, yva) - min(_h["val"])) < 1e-6, \
    "Al terminar, la red debería tener los pesos de su mejor época. ¿Guardaste una COPIA (copy.deepcopy) en `mejor_pesos`?"
print("✔ `entrenar` funciona y restaura la mejor época")

### Entrenar las tres redes

Cada red parte de la misma semilla. Tarda unos segundos por red.

In [ ]:
modelos, historiales, filas = {}, {}, []
for tipo in ["rnn", "lstm", "gru"]:
    fijar_semillas(SEED)
    m = Recurrente(tipo)
    t0 = time.time()
    historiales[tipo] = entrenar(m, Xtr, ytr, Xva, yva)
    modelos[tipo] = m
    filas.append({"modelo": tipo.upper(), "parámetros": n_parametros(m),
                  "MAE val": evaluar(m, Xva, yva)[0], "MAE prueba": evaluar(m, Xte, yte)[0],
                  "mejor época": historiales[tipo]["mejor_epoca"] + 1, "segundos": time.time() - t0})

# «Mañana = hoy» con LOS MISMOS objetivos que las redes: el último valor de cada ventana
filas.append({"modelo": "mañana = hoy", "parámetros": 0,
              "MAE val": mae(yva * sd, Xva[:, -1, 0] * sd), "MAE prueba": mae(yte * sd, Xte[:, -1, 0] * sd),
              "mejor época": None, "segundos": 0.0})
tabla = pd.DataFrame(filas).set_index("modelo")
tabla["mejor época"] = tabla["mejor época"].astype("Int64")
display(tabla.round(3))

base_te = tabla.loc["mañana = hoy", "MAE prueba"]
for nombre in ["RNN", "LSTM", "GRU"]:
    print(f"{nombre:>4}: baja el error de prueba un {1 - tabla.loc[nombre, 'MAE prueba'] / base_te:.0%} respecto de «mañana = hoy»")

In [ ]:
# Curvas de validación (el punto marca la mejor época) y predicción del mejor modelo en prueba
fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.5))
for tipo, h in historiales.items():
    ejes[0].plot(range(1, len(h["val"]) + 1), h["val"], color=COLOR_RED[tipo], label=tipo.upper())
    ejes[0].plot(h["mejor_epoca"] + 1, h["val"][h["mejor_epoca"]], "o", color=COLOR_RED[tipo], ms=8, mec="white")
ejes[0].set(title="Pérdida de validación (● = mejor época)", xlabel="época", ylabel="MSE (datos escalados)")
ejes[0].legend(frameon=False)

mejor = tabla.drop(index="mañana = hoy")["MAE val"].idxmin().lower()
pred = predecir(modelos[mejor], Xte)
ejes[1].plot(desescalar(yte), color=GRIS, lw=1.3, label="real")
ejes[1].plot(desescalar(pred), color=COLOR_RED[mejor], lw=1.3, label=f"predicción {mejor.upper()}")
ejes[1].set(title=f"Prueba: {mejor.upper()} frente a lo real", xlabel="paso (tramo de prueba)", ylabel="valor")
ejes[1].set_ylim(top=ejes[1].get_ylim()[1] + 0.7)
ejes[1].legend(loc="upper left", frameon=False, ncol=2)
plt.tight_layout()
plt.show()
print("Mejor modelo en validación:", mejor.upper())

In [ ]:
# --- Autochequeo A3 ---
assert set(tabla.index) == {"RNN", "LSTM", "GRU", "mañana = hoy"}, "La tabla debería tener RNN, LSTM, GRU y «mañana = hoy»."
for tipo, m in modelos.items():
    assert abs(perdida_val(m, Xva, yva) - min(historiales[tipo]["val"])) < 1e-6, \
        (f"{tipo.upper()} no quedó con los pesos de su mejor época. Revisa el final de `entrenar`: "
         "¿guardaste una COPIA de los pesos (copy.deepcopy(modelo.state_dict()))?")
if RAPIDO:
    print("⚠ RAPIDO = True: me salto el chequeo de calidad (con pocas épocas las redes todavía no alcanzan a aprender).")
else:
    redes = tabla.loc[["RNN", "LSTM", "GRU"], "MAE val"]
    assert (redes < tabla.loc["mañana = hoy", "MAE val"]).all(), (
        "Alguna red no le ganó a «mañana = hoy» en validación. Revisa que `entrenar` use lr=3e-3 "
        "y que al final restaure los pesos de la mejor época.")
print("✔ A3 OK: las tres redes le ganan a «mañana = hoy» en validación" if not RAPIDO else "✔ A3 OK (modo rápido)")

**Pregunta A3.** ¿Las redes le ganaron a «mañana = hoy»? ¿Por cuánto? ¿Por qué esa regla simple es una rival difícil en una serie con ruido?

**Respuesta:** (escribe aquí)

---
## Parte A4 — Pronóstico de 100 pasos y cuánta historia mirar (10 min)

> **En palabras simples (diapos 6 y 11).** Es como pronosticar el tiempo para los próximos 100 días usando tus propios pronósticos: cada día te equivocas un poco más.

**Consigna**

1. **Pronóstico recursivo.** La red predice un paso, lo agrega a su ventana como si fuera real y vuelve a predecir. Así, sus propios errores se van acumulando. Parte de una ventana real del tramo de prueba y avanza 100 pasos.
2. Grafica el pronóstico frente a lo real y mira cómo crece el error.
3. **¿Cuánta historia mirar?** Entrena las tres redes con ventanas de **5, 30 y 100** pasos y compara el error en prueba.

In [ ]:
def pronostico_recursivo(modelo, ventana_inicial, pasos=100):
    """Alimenta a la red con sus propias predicciones. `ventana_inicial`: (ventana,) escalada."""
    ventana = np.array(ventana_inicial, dtype=np.float32)
    salidas = []
    modelo.eval()
    with torch.no_grad():
        for _ in range(pasos):
            # TODO: 1) predice el valor siguiente a partir de `ventana`
            #          (la red espera forma (1, ventana, 1): torch.tensor(ventana[None, :, None]); .item() da el número)
            #       2) guarda la predicción en `salidas`
            #       3) corre la ventana: saca el valor más antiguo y agrega la predicción al final
            #          (pista: np.append(ventana[1:], np.float32(y)))
            ...
    return np.array(salidas)

In [ ]:
# --- Autochequeo: pronóstico recursivo ---
_p = pronostico_recursivo(modelos[mejor], test_e[:VENTANA], pasos=5)
assert _p.shape == (5,), f"Con pasos=5 deberías obtener 5 pronósticos y obtuviste forma {_p.shape}."
assert np.all(np.isfinite(_p)), "El pronóstico tiene valores no finitos."
assert abs(_p[0] - predecir(modelos[mejor], test_e[None, :VENTANA, None].astype(np.float32))[0]) < 1e-5, \
    "El primer pronóstico debería ser igual a la predicción de un paso con la ventana real."
assert abs(_p[1] - _p[0]) > 1e-6, "Los pronósticos no cambian: ¿estás corriendo la ventana en cada paso?"
print("✔ Pronóstico recursivo bien")

In [ ]:
PASOS = 100
ventana0 = test_e[:VENTANA]
real_fut = test_e[VENTANA:VENTANA + PASOS]
pron = pronostico_recursivo(modelos[mejor], ventana0, PASOS)
quieto = np.full(PASOS, ventana0[-1])        # comparación: repetir para siempre el último valor conocido

err_red = np.abs(pron - real_fut) * sd
err_quieto = np.abs(quieto - real_fut) * sd

fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.5))
ejes[0].plot(range(-VENTANA, 0), desescalar(ventana0), color=NARANJA, lw=1.3, label="ventana inicial (real)")
ejes[0].plot(range(PASOS), desescalar(real_fut), color=GRIS, lw=1.3, label="lo que pasó")
ejes[0].plot(range(PASOS), desescalar(pron), color=COLOR_RED[mejor], lw=1.5, label=f"pronóstico {mejor.upper()}")
ejes[0].set(title="Pronóstico recursivo a 100 pasos", xlabel="pasos hacia adelante", ylabel="valor")
ejes[0].set_ylim(top=ejes[0].get_ylim()[1] + 0.9)
ejes[0].legend(loc="upper left", frameon=False, fontsize=9, ncol=3)
ejes[1].plot(err_red, color=COLOR_RED[mejor], label=mejor.upper())
ejes[1].plot(err_quieto, color=GRIS, label="repetir el último valor")
ejes[1].set(title="Error según qué tan lejos pronosticamos", xlabel="pasos hacia adelante", ylabel="|error|")
ejes[1].legend(frameon=False)
plt.tight_layout()
plt.show()
un_paso = predecir(modelos[mejor], Xte[:PASOS])   # comparación: predecir un solo paso, siempre con ventanas reales
print(f"MAE del pronóstico recursivo: pasos 1–10 = {err_red[:10].mean():.3f} | pasos 91–100 = {err_red[-10:].mean():.3f} "
      f"| los 100 pasos = {err_red.mean():.3f}")
print(f"En esos mismos 100 pasos: predecir un solo paso con ventanas reales = {mae(real_fut * sd, un_paso * sd):.3f} "
      f"| repetir el último valor = {err_quieto.mean():.3f}")

### ¿Cuánta historia conviene mirar?

Las tres redes con ventanas de 5, 30 y 100 pasos, con la misma receta. Son 9 entrenamientos cortos (alrededor de un minuto en total). La tabla muestra el **MAE en prueba**.

In [ ]:
filas_v = []
for v in [5, 30, 100]:
    Xa, ya = crear_ventanas(train_e, v)
    Xb, yb = crear_ventanas(val_e, v)
    Xc, yc = crear_ventanas(test_e, v)
    fila = {"ventana": v}
    for tipo in ["rnn", "lstm", "gru"]:
        fijar_semillas(SEED)
        m = Recurrente(tipo)
        entrenar(m, Xa, ya, Xb, yb)
        fila[tipo.upper()] = evaluar(m, Xc, yc)[0]
    fila["mañana = hoy"] = mae(yc * sd, Xc[:, -1, 0] * sd)
    filas_v.append(fila)
tabla_v = pd.DataFrame(filas_v).set_index("ventana")
display(tabla_v.round(3))

**Pregunta A4.** ¿Qué pasa con el error a medida que el pronóstico se aleja? ¿Por qué una ventana de 5 pasos funciona peor que una de 30?

**Respuesta:** (escribe aquí)

In [ ]:
print(f"Tiempo total del notebook: {time.time() - T0:.0f} s")

---
## Para saber más (opcional): las fórmulas, en palabras

Nada de esto es necesario para el lab. Es lo mismo del Anexo A del deck (diapo 20).

**RNN simple**

$$h_t = \tanh\left(W_x\,x_t + W_h\,h_{t-1} + b\right)$$

*En palabras:* la memoria nueva ($h_t$) mezcla lo que entra ahora ($x_t$, el valor de este paso) con la memoria anterior ($h_{t-1}$). $W_x$ y $W_h$ dicen cuánto pesa cada cosa, $b$ es un ajuste fijo y $\tanh$ aplasta el resultado entre −1 y 1. Los **mismos** pesos se usan en todos los pasos.

**LSTM** (la autopista con válvulas de la diapo 7). En cada paso mira $[h_{t-1}, x_t]$: la memoria anterior y el valor nuevo.

| Fórmula | En palabras |
|---|---|
| $f_t = \sigma(W_f\,[h_{t-1}, x_t] + b_f)$ | **Válvula de olvido:** de 0 a 1, cuánto conservar de cada apunte viejo. |
| $i_t = \sigma(W_i\,[h_{t-1}, x_t] + b_i)$ | **Válvula de entrada:** de 0 a 1, cuánto anotar de lo nuevo. |
| $\tilde c_t = \tanh(W_c\,[h_{t-1}, x_t] + b_c)$ | **Lo que se propone anotar.** |
| $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$ | **La libreta (la autopista):** lo que se conserva + lo nuevo que se anota. Como aquí se *suma*, la información viaja lejos sin achicarse en cada paso. |
| $o_t = \sigma(W_o\,[h_{t-1}, x_t] + b_o)$ | **Válvula de salida:** qué parte de la libreta mostrar. |
| $h_t = o_t \odot \tanh(c_t)$ | **La memoria que se muestra** (y la que usa la capa lineal en el último paso). |

$\sigma$ es la sigmoide (da números entre 0 y 1, como una válvula) y $\odot$ es multiplicar número a número. La **GRU** hace lo mismo con dos válvulas en vez de tres y sin libreta aparte.

**Curiosidad: los 1.153 parámetros de la RNN de este lab**

- Capa recurrente: $32\cdot 1$ (pesos de la entrada) $+\ 32\cdot 32$ (pesos de la memoria) $+\ 32 + 32$ (dos ajustes fijos; PyTorch usa dos) $= 1.120$.
- Capa lineal: $32$ pesos $+\ 1$ ajuste $= 33$.
- Total: $1.120 + 33 = 1.153$. La LSTM repite la capa recurrente 4 veces (una por válvula más la propuesta): $4 \cdot 1.120 + 33 = 4.513$. La GRU, 3 veces: $3 \cdot 1.120 + 33 = 3.393$.